In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, auc


: 

In [218]:
#1 - Calcular médias móveis de transações nos últimos 3, 6 e 12 meses por cliente
df = pd.read_csv('transacoes_clientes.csv')

In [ ]:
#print(df.loc[df['id_cliente'] == 488])
#df_cliente_mes = df.loc[(df['id_cliente'] == 488) & (df['mes'].isin([3, 6, 12]))]
#print(df_cliente_mes)
#print("Média: " + str(df_cliente_mes['valor_total'].mean()))

filtro = df.loc[df['mes'].isin([3, 6, 12])]
print(filtro.groupby('id_cliente')[['valor_total']].mean().sort_values(by='id_cliente', ascending=True))

In [ ]:
#2- Realizar merge entre bases de perfiç, transações e reclamações em um único dataframe
df_clientes = pd.read_csv('clientes.csv')
df_transacoes = pd.read_csv('transacoes_clientes.csv')
df_reclamacoes = pd.read_csv('reclamacoes.csv')

df_transacoes = df_transacoes.sort_values(['id_cliente', 'mes'], ascending=True)
#print(df_transacoes.head())

df_concatenado = pd.concat([df_clientes, df_transacoes, df_reclamacoes], axis=1, ignore_index=False)
print(df_concatenado.head())


### Gere um gráfico de barras com a taxa de churn por tipo de produto (cartão, conta, investimento)


In [221]:
#3 - Gere um gráfico de barras com a taxa de churn por tipo de produto (cartão, conta, investimento)

#Criando a coluna tipo_produto 
tipos = ['cartao', 'conta', 'investimento']

df_clientes['tipo_produto'] = np.random.choice(tipos, size=500)

In [222]:
#Criando a coluna churn para o exemplo
# Definindo a probabilidade base
df_concatenado['probabilidade'] = 0.1  # Probabilidade inicial de 10%

# Regra: Reclamações não resolvidas aumentam o risco em 3x 
df_concatenado.loc[df_concatenado['resolvido'] == 0, 'probabilidade'] *= 3

# Regra: Baixa atividade (ex: menos de 20 transações) aumenta o risco 
df_concatenado.loc[df_concatenado['quantidade_transacoes'] < 20, 'probabilidade'] += 0.2

# Criando a coluna Churn binária (Target) baseada na probabilidade calculada
# Isso garante que o target seja entre 15% e 20% conforme o requisito [cite: 21]
df_concatenado['churn'] = (df_concatenado['probabilidade'] > 0.5).astype(int)

In [ ]:
df_clientes.head()

In [ ]:
taxa_churn = df_clientes.groupby('tipo_produto')['churn'].mean()
print(taxa_churn)

In [ ]:
# Plotar o gráfico de barras
taxa_churn.plot(kind='bar')
plt.title('Taxa de Churn por Tipo de Produto')
plt.xlabel('Tipo de Produto')
plt.ylabel('Taxa de Churn')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Gerar regplot mostrando a relação entre tempo de relacionamento e probabilidade de churn

In [ ]:
# Identificar clientes ativos no mês 12
clientes_ativos_mes12 = df_transacoes[df_transacoes['mes'] == 12]['id_cliente'].unique()

# Número total de clientes
total_clientes = len(df_clientes)

# Número de clientes ativos no mês 12
num_ativos = len(clientes_ativos_mes12)

# Taxa de churn (clientes não ativos no mês 12 / total de clientes)
taxa_churn_mes12 = ((total_clientes - num_ativos) / total_clientes ) * 100

print(f"Total de clientes: {total_clientes}")
print(f"Clientes ativos no mês 12: {num_ativos}")
print(f"Taxa de churn baseada na atividade no mês 12: {taxa_churn_mes12:.4f}")

df_clientes['churn_mes'] = np.where(df_clientes['id_cliente'].isin(clientes_ativos_mes12), 0, 1)

sns.regplot(
    data=df_clientes,
    x='tempo_relacionamento_meses',
    y='churn_mes'
)

plt.title('Tempo de Relacionamento x Churn')
plt.xlabel('Tempo de Relacionamento (meses)')
plt.ylabel('Churn')

plt.show()

### Treinar um modelo de Regressão Logística com AUC -ROC >= 0,80 para prever churn

In [ ]:
df_concatenado = pd.concat([df_clientes, df_transacoes, df_reclamacoes], axis=1, ignore_index=False)
df_concatenado.head()

In [228]:
x = df_concatenado[['id_cliente', 'tempo_relacionamento_meses', 'mes', 'numero_produtos', 'quantidade_transacoes', 'valor_total', 'resolvido']]
y = df_concatenado['churn']

In [229]:
x = x.replace(np.nan, 0, inplace=True)
y = y.replace(np.nan, 0, inplace=True)

In [ ]:
x

In [231]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
x_train

In [ ]:
modelo = LogisticRegression()
modelo.fit(x_train, y_train)

In [ ]:
y_probs = modelo.predict_proba(x_test)[:, 1]

In [ ]:
# Calcular FPR, TPR e thresholds
auc = roc_auc_score(y_test, y_probs)

# Calcular a área sob a curva (AUC)
#valor_auc = auc(fpr, tpr)
print(f"O valor do AUC é: {auc:.4f}")

# RPA

In [ ]:


# Criando a pasta de relatórios caso não exista 
if not os.path.exists('./relatorios_churn'):
    os.makedirs('./relatorios_churn')

# 1. Identificando os Top 20 clientes com maior probabilidade de churn 
df_concatenado['probabilidade_modelo'] = modelo.predict_proba(scaler.transform(X))[:, 1]
top_20_risco = df_concatenado.nlargest(20, 'probabilidade_modelo')

# 2. Criando colunas de "Ação Sugerida" e "Motivo" para o RPA 
top_20_risco['Ação Sugerida'] = "Oferecer isenção de tarifa e ligar para o cliente"
top_20_risco['Motivo Principal'] = top_20_risco['motivo'].fillna("Baixo Engajamento")

# 3. Selecionando apenas as colunas pedidas no PDF 
relatorio_final = top_20_risco[['id_cliente', 'probabilidade_modelo', 'Motivo Principal', 'Ação Sugerida']]

# 4. Salvando em Excel 
caminho_arquivo = './relatorios_churn/relatorio_diario_churn.xlsx'
relatorio_final.to_excel(caminho_arquivo, index=False)

print(f"RPA Concluído: Relatório salvo em {caminho_arquivo}")